# MentalHealthAI: huấn luyện lại và đo trên GPU (Bước 1-2)

Chạy **từ trên xuống**, mỗi ô một lần. Chọn `Runtime > Change runtime type > GPU (T4)` trước khi bắt đầu.

| Phần | Làm gì | Ước lượng thời gian (T4, ghi chú: chỉ là ước lượng, thời gian thật phụ thuộc cỡ dữ liệu) |
|---|---|---|
| 0 | Kiểm tra GPU, lấy mã, cài thư viện | vài phút |
| 1 | Đưa dữ liệu vào, làm sạch, baseline TF-IDF | vài phút |
| 2 | **Bước 1:** đo lại 3 mô hình hiện có trên tập test | 10-20 phút |
| 3 | **Bước 2:** Depression, nhiều seed, so với baseline | 1-3 giờ |
| 4 | **Bước 3 (tùy chọn):** huấn luyện lại Stress nhiều seed | 1-2 giờ |
| 5 | Gói kết quả tải về | vài phút |

**Quy tắc để kết quả đáng tin:** tập test chỉ dùng để báo cáo, không dùng để chọn mô hình hay ngưỡng (đã có tập dev riêng).
Mọi con số báo cáo phải là trung bình ± độ lệch chuẩn của nhiều seed, và nếu BERT không vượt baseline thì ghi đúng như vậy.

## 0. Chuẩn bị

In [ ]:
import subprocess, sys, os
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout)
import torch
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available())
assert torch.cuda.is_available(), "Chưa bật GPU: Runtime > Change runtime type > GPU"

In [ ]:
# Lấy mã nguồn. Nếu repo ở chế độ private: dùng token cá nhân (quyền đọc) và TUYỆT ĐỐI không chia sẻ notebook đã chứa token.
REPO_URL = "https://github.com/DucThinhmta/MentalHealthAI.git"
BRANCH = "main"
if not os.path.exists("MentalHealthAI"):
    !git clone --branch $BRANCH $REPO_URL
%cd MentalHealthAI
!git log --oneline | head -3
!pip -q install -r requirements.txt

## 1. Dữ liệu

Repo **không chứa dữ liệu** (xem `data/README.md`). Đưa các tệp sau vào thư mục Google Drive `MyDrive/MentalHealthAI_data/`:

- Stress (Dreaddit): `train_clean.csv`, `test_clean.csv` (cột `text_clean`, `label`)
- Emotion (GoEmotions): `train_emotion.csv`, `dev_emotion.csv`, `test_emotion.csv`
- Depression (Depression Severity Dataset): `depression_train.csv`, `depression_test.csv` (cột `text`, `label` 0-3)

Đọc điều khoản sử dụng của từng bộ dữ liệu trước khi dùng.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import shutil, pathlib
SRC_DIR = pathlib.Path("/content/drive/MyDrive/MentalHealthAI_data")
NEEDED = ["train_clean.csv", "test_clean.csv", "train_emotion.csv", "dev_emotion.csv", "test_emotion.csv",
          "depression_train.csv", "depression_test.csv"]
pathlib.Path("data").mkdir(exist_ok=True)
missing = []
for name in NEEDED:
    if (SRC_DIR / name).exists():
        shutil.copy(SRC_DIR / name, pathlib.Path("data") / name)
    else:
        missing.append(name)
print("Thiếu:", missing or "không thiếu tệp nào")

In [ ]:
!python src/clean_data.py
!python src/baseline_tfidf.py
!cat reports/baselines.csv

## 2. Bước 1: đo lại 3 mô hình hiện có trên tập test

Tải 3 mô hình đã huấn luyện từ Hugging Face (nếu private: chạy `!hf auth login` trước) rồi đo. Đây là con số "trước khi cải thiện".

In [ ]:
!python src/download_models.py
!python src/evaluate_all.py --task all
!python src/calibrate.py

In [ ]:
import json
print(json.dumps(json.load(open("reports/results.json")), indent=1, ensure_ascii=False)[:3000])

## 3. Bước 2: sửa nhánh Depression (BERT đang thua TF-IDF về macro-F1)

Hai cấu hình, mỗi cấu hình 3 seed, so theo từng seed:

- `base`: cấu hình đối chứng (hàm mất mát thường)
- `focal`: focal loss + làm dịu trọng số lớp + label smoothing + chọn epoch theo QWK (nhắm vào lớp hiếm `mild`, `severe`)

Có thể thêm cấu hình thứ ba bằng `--base-model roberta-base`. Nếu hết bộ nhớ GPU, thêm `--batch-size 8` vào phần sau dấu `--`.

In [ ]:
SEEDS = "42 43 44"
!python src/run_seeds.py --task depression --tag base  --seeds $SEEDS -- --epochs 4 --patience 2
!python src/run_seeds.py --task depression --tag focal --seeds $SEEDS -- --loss focal --weight-power 0.5 --label-smoothing 0.05 --metric qwk --epochs 6 --patience 2

In [ ]:
!python src/run_seeds.py --compare focal base --task depression --seeds $SEEDS
!python src/compare_depression.py

**Đọc kết quả:** ô trên in kết luận theo quy tắc đặt trước (xem đầu `src/compare_depression.py`). Ba khả năng:

1. *BERT vượt baseline*: dùng cấu hình đó, ghi khoảng tin cậy vào `docs/MODEL_CARD.md`.
2. *Baseline vượt BERT*: dùng TF-IDF + LogReg cho nhánh Depression (hoặc ghép hai mô hình) và **ghi đúng như vậy** trong báo cáo. Đây vẫn là một kết quả tốt.
3. *Chưa phân biệt được*: tăng lên 5-10 seed rồi chạy lại, hoặc báo cáo cả hai mà không khẳng định bên nào tốt hơn.

## 4. (Tùy chọn) Stress nhiều seed

Hiện chỉ có con số một lần chạy (accuracy 0,794). Chạy 3 seed để có trung bình ± độ lệch chuẩn.

In [ ]:
!python src/run_seeds.py --task stress --tag base --seeds $SEEDS -- --epochs 4 --patience 2

## 5. Gói kết quả để tải về

Mang `reports/` và (nếu muốn dùng) mô hình tốt nhất về máy. **Không đẩy dữ liệu lên Git.**

In [ ]:
!zip -qr results_$(date +%Y%m%d).zip reports models/seeds -x "*.safetensors" -x "*.bin" -x "*optimizer*"
from google.colab import files
files.download(sorted(pathlib.Path(".").glob("results_*.zip"))[-1].name)

In [ ]:
# Chỉ chạy nếu muốn lưu mô hình tốt nhất vào Drive (mô hình nặng):
# !cp -r models/seeds/depression_focal_s42 /content/drive/MyDrive/MentalHealthAI_models/